# Purpose

The goal of this notebook is to combine all the model `metrics.csv` into 1 table.

# Import

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Pathing

In [2]:
MODELING_PATH = Path("../../data/processed/modeling")

# Create Metric Files

In [3]:
metric_files = {
    "linear_regression": MODELING_PATH / "linear_regression" / "metrics.csv",
    "ridge_regression": MODELING_PATH / "ridge_regression" / "metrics.csv",
    "random_forest": MODELING_PATH / "random_forest" / "metrics.csv",
    "gradient_boosting": MODELING_PATH / "gradient_boosting" / "metrics.csv",
    "garch_spy_only": MODELING_PATH / "garch" / "metrics.csv",
}

# Create Model Comparison

Had to bring back `evaluation_predictions` to recalculate the Random Forest as numbers cols were off

In [4]:
def evaluate_predictions(prediction_path, prediction_col, actual_col="future_volatility_20d"):
    predictions = pd.read_csv(prediction_path)

    y_true = predictions[actual_col].astype(float)
    y_pred = predictions[prediction_col].astype(float)

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2 = 1 - (((y_true - y_pred) ** 2).sum() / ((y_true - y_true.mean()) ** 2).sum())

    return mae, rmse, r2

In [5]:
all_metrics = []

for model_folder, file_path in metric_files.items():
    metrics = pd.read_csv(file_path)

    if model_folder == "random_forest":
        rf_mae, rf_rmse, rf_r2 = evaluate_predictions(
            MODELING_PATH / "random_forest" / "test_predictions.csv",
            "predicted_future_volatility_20d"
        )

        metrics = pd.DataFrame([
            {
                "model": "Baseline: rolling_volatility_20d",
                "MAE": 0.0046471957325684545,
                "RMSE": 0.007130185107893034,
                "R2": 0.0015102059479311647
            },
            {
                "model": "Random Forest (tuned)",
                "MAE": rf_mae,
                "RMSE": rf_rmse,
                "R2": rf_r2
            }
        ])

    metrics["source_folder"] = model_folder
    metrics["scope"] = "SPY only" if model_folder == "garch_spy_only" else "All tickers"
    all_metrics.append(metrics)

In [6]:
raw_model_comparison = pd.concat(all_metrics, ignore_index=True)

raw_model_comparison

,model,MAE,RMSE,R2,source_folder,scope
0,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,linear_regression,All tickers
1,Linear Regression,0.003941,0.005984,0.296699,linear_regression,All tickers
2,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,ridge_regression,All tickers
3,Linear Regression,0.003941,0.005984,0.296699,ridge_regression,All tickers
4,Ridge Regression,0.003943,0.005984,0.296628,ridge_regression,All tickers
5,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,random_forest,All tickers
6,Random Forest (tuned),0.003768,0.005808,0.337592,random_forest,All tickers
7,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,gradient_boosting,All tickers
8,Gradient Boosting,0.004004,0.006071,0.276082,gradient_boosting,All tickers
9,Baseline: rolling_volatility_20d,0.004018,0.006628,-0.419243,garch_spy_only,SPY only


# Clean All-Ticker Model Comparison

This table keeps only the all-ticker models and removes duplicate baseline or repeated comparison rows.

In [7]:
all_ticker_model_comparison = raw_model_comparison[
    raw_model_comparison["scope"] == "All tickers"
].copy()

all_ticker_model_comparison = all_ticker_model_comparison.drop_duplicates(
    subset=["model"],
    keep="first"
)

all_ticker_model_comparison = all_ticker_model_comparison[
    ["model", "MAE", "RMSE", "R2", "source_folder", "scope"]
].sort_values("RMSE")

all_ticker_model_comparison

,model,MAE,RMSE,R2,source_folder,scope
6,Random Forest (tuned),0.003768,0.005808,0.337592,random_forest,All tickers
1,Linear Regression,0.003941,0.005984,0.296699,linear_regression,All tickers
4,Ridge Regression,0.003943,0.005984,0.296628,ridge_regression,All tickers
8,Gradient Boosting,0.004004,0.006071,0.276082,gradient_boosting,All tickers
0,Baseline: rolling_volatility_20d,0.004647,0.007130,0.001510,linear_regression,All tickers


# SPY-Only Statistical Model Comparison

GARCH is shown separately because this notebook only built the statistical model for `SPY`, not every ticker.

In [8]:
garch_spy_comparison = raw_model_comparison[
    raw_model_comparison["scope"] == "SPY only"
].copy()

garch_spy_comparison = garch_spy_comparison[
    ["model", "MAE", "RMSE", "R2", "source_folder", "scope"]
].sort_values("RMSE")

garch_spy_comparison

,model,MAE,RMSE,R2,source_folder,scope
10,"GARCH(1,1)",0.003815,0.006027,-0.173700,garch_spy_only,SPY only
9,Baseline: rolling_volatility_20d,0.004018,0.006628,-0.419243,garch_spy_only,SPY only


# Save Model Comparison Outputs

In [9]:
MODEL_COMPARISON_PATH = MODELING_PATH / "model_comparison"
MODEL_COMPARISON_PATH.mkdir(parents=True, exist_ok=True)

all_ticker_model_comparison.to_csv(
    MODEL_COMPARISON_PATH / "all_ticker_model_comparison_metrics.csv",
    index=False
)

garch_spy_comparison.to_csv(
    MODEL_COMPARISON_PATH / "garch_spy_comparison_metrics.csv",
    index=False
)

raw_model_comparison.to_csv(
    MODEL_COMPARISON_PATH / "raw_model_comparison_metrics.csv",
    index=False
)

print("Saved:", MODEL_COMPARISON_PATH / "all_ticker_model_comparison_metrics.csv")
print("Saved:", MODEL_COMPARISON_PATH / "garch_spy_comparison_metrics.csv")
print("Saved:", MODEL_COMPARISON_PATH / "raw_model_comparison_metrics.csv")

Saved: ..\..\data\processed\modeling\model_comparison\all_ticker_model_comparison_metrics.csv
Saved: ..\..\data\processed\modeling\model_comparison\garch_spy_comparison_metrics.csv
Saved: ..\..\data\processed\modeling\model_comparison\raw_model_comparison_metrics.csv


# Model Comparison Takeaway

For the all-ticker volatility prediction models, Random Forest has the lowest RMSE and highest R2, so it is currently the strongest predictive model. Linear Regression and Ridge Regression perform almost the same, which suggests the original linear model was already fairly stable. Gradient Boosting still improves on the baseline, but it does not outperform the linear models or Random Forest.

The GARCH model is kept separate because it was only trained and evaluated on `SPY`. It is useful as a statistical volatility benchmark, but it should not be ranked directly against the all-ticker machine learning models.